### Objetivo do Projeto

#### Analisar o desempenho de vendas de uma empresa do setor alimentício, identificando:

- evolução do faturamento;
- crescimento anual;
- categorias mais lucrativas;
- comportamento das vendas ao longo do tempo.

##### Tecnologias utilizadas:

- Python
- Pandas
- Plotly

### Etapa 1. Importação das Bibliotecas

In [362]:
import pandas as pd
from pathlib import Path
import plotly.express as px

### Etapa 2. Configurações

In [363]:
# Cria a pasta output se ainda não existir
out = Path('output')
out.mkdir(exist_ok=True)

### Etapa 3. Carregamento e Visualização dos Dados

In [364]:
df_vendas = pd.read_csv(out/'Vendas_Chocolate.csv', sep=';')


print(f'Total de linhas: {len(df_vendas)}\n')

print('Valores nullos em cada coluna:')
display(df_vendas.isnull().sum())

print('Tipos de dados em cada coluna:')
display(df_vendas.dtypes)

display(df_vendas.head())

Total de linhas: 3282

Valores nullos em cada coluna:


Vendedor             32
País                 32
Produto              32
Data                  0
Valor                 0
Caixas Enviadas      32
Data de Hoje       3281
dtype: int64

Tipos de dados em cada coluna:


Vendedor            object
País                object
Produto             object
Data                object
Valor               object
Caixas Enviadas    float64
Data de Hoje        object
dtype: object

,Vendedor,País,Produto,Data,Valor,Caixas Enviadas,Data de Hoje
0,João Rocha,Reino Unido,Chocolate com Menta e Flocos,04/01/2022,"$5,320.00",180.0,03/02/2026
1,Vanderlei Teixeira,Índia,NaN,01/08/2022,"$7,896.00",94.0,NaN
2,Gisele Barbosa,Índia,Cubos de Manteiga de Amendoim,07/07/2022,"$4,501.00",91.0,NaN
3,Janaína Moreira,Austrália,Cubos de Manteiga de Amendoim,27/04/2022,"$12,726.00",342.0,NaN
4,João Rocha,Reino Unido,Cubos de Manteiga de Amendoim,24/02/2022,"$13,685.00",184.0,NaN


### Etapa 4. Limpeza e Tratamento

In [365]:
# Excluir colunas desnecessárias:
colunas_desnecessarias = ['Data de Hoje'] # Criar uma lista com os nomes das colunas para exclusão
df_vendas = df_vendas.drop(columns=colunas_desnecessarias)

# Excluir valores nullos:
df_vendas = df_vendas.dropna()

# Excluir valores duplicados:
df_vendas = df_vendas.drop_duplicates()

# Corrigir o Dtype das colunas:
# 1.formata para dia/mes/ano
df_vendas['Data'] = pd.to_datetime(df_vendas['Data'],dayfirst=True) 

# 2.Formatar a coluna valor para Float:
df_vendas['Valor'] = (
    df_vendas['Valor']
    .str.replace("$",'', regex=False)
    .str.replace(",","")
    .astype("float")
)

### Etapa Opcional para validação da Base de dados

In [366]:
# print(f'Total de linhas: {len(df_vendas)}\n')

print('Valores nullos em cada coluna:')
display(df_vendas.isnull().sum())

print('Tipos de dados em cada coluna:')
display(df_vendas.dtypes)

display(df_vendas.head())

# Formata os valores do Describe, para 2 casas decimais os valores forem int ou flot
resultado = df_vendas.describe()
resultado_formatado = resultado.applymap(lambda x: f'{x:.2f}' if isinstance(x, (float, int)) else x)
display(resultado_formatado)

Valores nullos em cada coluna:


Vendedor           0
País               0
Produto            0
Data               0
Valor              0
Caixas Enviadas    0
dtype: int64

Tipos de dados em cada coluna:


Vendedor                   object
País                       object
Produto                    object
Data               datetime64[ns]
Valor                     float64
Caixas Enviadas           float64
dtype: object

,Vendedor,País,Produto,Data,Valor,Caixas Enviadas
0,João Rocha,Reino Unido,Chocolate com Menta e Flocos,2022-01-04,5320.0,180.0
2,Gisele Barbosa,Índia,Cubos de Manteiga de Amendoim,2022-07-07,4501.0,91.0
3,Janaína Moreira,Austrália,Cubos de Manteiga de Amendoim,2022-04-27,12726.0,342.0
4,João Rocha,Reino Unido,Cubos de Manteiga de Amendoim,2022-02-24,13685.0,184.0
5,Vanderlei Teixeira,Índia,Salgadinho Suave e Sedoso,2022-06-06,5376.0,38.0


C:\Users\SANTIAGO\AppData\Local\Temp\ipykernel_20120\4175659633.py:13: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



,Data,Valor,Caixas Enviadas
count,3155,3155.00,3155.00
mean,2023-05-01 17:10:08.177496064,6021.79,164.11
min,2022-01-03 00:00:00,7.00,1.00
25%,2022-06-30 00:00:00,2510.81,71.00
50%,2023-05-10 00:00:00,5220.80,137.00
75%,2024-03-01 12:00:00,8548.92,232.00
max,2024-08-31 00:00:00,26170.95,778.00
std,nan,4393.27,123.29


### Etapa 5. Engenharia de Dados

In [367]:
# Analise de faturamento ao longo do tempo:

# Cria as colunas adicionais de data:
df_vendas["Numero Mes"] = df_vendas["Data"].dt.month
df_vendas['Ano'] = df_vendas["Data"].dt.year
df_vendas["Mes/Ano"] = df_vendas["Data"].dt.to_period("M").astype("str")


In [368]:
# Definir as categorias para os tipos de produtos
categorias = {
    '99% Amargo e Puro': 'Amargo',
    'Amêndoas Cobertas com Chocolate': 'Amêndoas',
    'Barras 85% Cacau': 'Barras',
    'Barras Recheadas com Caramelo': 'Barras',
    'Barras de Chocolate ao Leite': 'Barras',
    'Barras de Frutas e Castanhas': 'Barras',
    'Bombas de Chocolate': 'Bombas',
    'Chocolate Branco': 'Chocolate',
    'Chocolate Solúvel': 'Chocolate',
    'Chocolate com Amêndoas': 'Chocolate',
    'Chocolate com Mel de Manuka': 'Chocolate',
    'Chocolate com Menta e Flocos': 'Chocolate',
    'Chocolate de Framboesa': 'Chocolate',
    'Chocolate de Laranja': 'Chocolate',
    'Cubos de Manteiga de Amendoim': 'Cubos',
    'Depois das Nove': 'Especiais',
    'Especiais Apimentados Finos': 'Especiais',
    'Gotas de Chocolate de Confeiteiro': 'Gotas',
    'Mordidas 50% Cacau': 'Mordidas',
    'Mordidas 70% Cacau': 'Mordidas',
    'Salgadinho Suave e Sedoso': 'Salgadinho',
    'Xarope de Chocolate Orgânico': 'Xarope'
}

# Aplicar a categorização aos produtos
df_vendas['Categoria'] = df_vendas['Produto'].apply(lambda produto: categorias.get(produto, 'Outros'))

### Etapa 6. Análise Exploratória

In [369]:
# Cria agrupamentos, para análise
columas_vendas_mes_ano = ["Mes/Ano","Valor"]
vendas_mes = df_vendas[columas_vendas_mes_ano].groupby("Mes/Ano", as_index=False).sum()
display(vendas_mes.style.format({
    "Valor" : "${:,.2f}"
}))

,Mes/Ano,Valor
0,2022-01,"$860,363.00"
1,2022-02,"$699,377.00"
2,2022-03,"$726,418.00"
3,2022-04,"$653,198.00"
4,2022-05,"$740,103.00"
5,2022-06,"$836,465.00"
6,2022-07,"$772,177.00"
7,2022-08,"$709,065.00"
8,2023-01,"$900,155.00"
9,2023-02,"$729,416.95"


In [370]:
# Análise de volume e Ticket médio:
colunas_vendas_ano = ["Ano","Valor","Caixas Enviadas"]
vendas_anual = df_vendas[colunas_vendas_ano]
vendas_anual = vendas_anual.groupby("Ano",as_index=False).sum()

vendas_anual["Ticket Médio"] = vendas_anual["Valor"] / vendas_anual["Caixas Enviadas"]
vendas_anual["%_Crescimento_Anual"] = vendas_anual["Valor"].pct_change()

display(vendas_anual.style.format({
    "Valor"                 : "${:,.1f}",
    "Caixas Enviadas"       : "{:,.0f}",
    "Ticket Médio"          : "${:,.1f}",
    "%_Crescimento_Anual"   : "{:,.1%}"
}))

,Ano,Valor,Caixas Enviadas,Ticket Médio,%_Crescimento_Anual
0,2022,"$5,997,166.0","171,422",$35.0,nan%
1,2023,"$6,396,868.9","173,970",$36.8,6.7%
2,2024,"$6,604,719.0","172,361",$38.3,3.2%


### Etapa 7. Gráficos

In [371]:
# Cria o Gráfico analisar a tendência ao longo do tempo:
grafico_faturament_mes_ano = px.line(
    vendas_mes, 
    x="Mes/Ano", 
    y="Valor",
    title="Evolução do Faturamento ao Longo do Tempo"
    )

# Centralizando as informações no gráfico:
grafico_faturament_mes_ano.update_layout(
    title_x=0.5,
    title_xanchor='center',
    yaxis_title="Valor (R$)",
    xaxis_title="Mês/Anos"
)
grafico_faturament_mes_ano.show()

In [374]:
# Análise de Faturamento da Categoria do Produtos por Ano:
lista_colunas = ["Ano", "Categoria", "Valor"]
faturamento_categoria_anual = df_vendas[lista_colunas]
faturamento_categoria_anual = faturamento_categoria_anual.groupby(["Ano","Categoria"], as_index=False).sum()

# Converte Ano para texto (string) para o eixo x ficar correto:
faturamento_categoria_anual["Ano"] = faturamento_categoria_anual["Ano"].astype(str)

# Cria um gráfico de barras para análisar
# a distribuição do faturamento das Categorias,
# ao longo dos anos:
grafico_faturamento_categoria_ano = px.bar(
    faturamento_categoria_anual, 
    x="Ano", 
    y="Valor",
    title="Faturamento por Categoria ao Longo dos Anos", 
    color = "Categoria", 
    barmode='group'
    )
# Centralizando as informações no gráfico:
grafico_faturamento_categoria_ano.update_layout(
    title_x = 0.5,
    title_xanchor = 'center',
    yaxis_title="Valor (R$)",
    xaxis_title="Ano",
    legend_title="Categoria",
)

# Força o eixo Y começar no 0:
grafico_faturamento_categoria_ano.update_yaxes(rangemode="tozero")

# Salvar o gráfico:
grafico_faturamento_categoria_ano.write_image("faturmento_por_categoria.png")

grafico_faturamento_categoria_ano.show()

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


### Etapa Final: Conclusão

##### Principais insights encontrados:

- crescimento consistente do faturamento ao longo do tempo;
- categorias (Chocolate e Barras) com maior representatividade nas vendas;
- evolução do ticket médio anual.